# 03 — Inner Loop: Hierarchical Weight Fitting

Given a fixed DSL structure S, fit the hierarchical weights:

$$\theta_{i,c} = \theta_g + \theta_c + \Delta_i$$

MAP objective minimized via L-BFGS-B:

$$\mathcal{L}(\theta) = -\sum \log P_\theta(a^* | s_t) + \frac{1}{2\sigma^2}\|\theta_g\|^2 + \frac{1}{2\tau^2}\sum_c \|\theta_c\|^2 + \frac{1}{2\nu^2}\sum_i \|\Delta_i\|^2$$

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pickle
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from src.dsl import DSLStructure
from src.inner_loop import run_inner_loop, compute_posterior_score
from src.evaluation import compute_metrics, compute_residuals, print_metrics
from src.checkpoint import Checkpoint

DATA_DIR = Path('../data')
ckpt = Checkpoint('../data')
print('Pipeline status:')
for stage, status in ckpt.status().items():
    print(f'  {stage:15s} {status}')

In [ ]:
with open(DATA_DIR / 'train_features.pkl', 'rb') as f:
    data = pickle.load(f)

features_list  = data['features_list']
chosen_indices = data['chosen_indices']
customer_ids   = data['customer_ids']
categories     = data['categories']
structure      = DSLStructure.from_dict(data['structure'])

print(f'Structure: {structure}')
print(f'Training events: {len(features_list):,}')

## Fit weights on initial structure

In [ ]:
weights, score = run_inner_loop(
    structure, features_list, chosen_indices, customer_ids, categories,
    sigma=10.0, tau=1.0, nu=0.5
)

print(f'Posterior score: {score:.2f}')
print('\nGlobal weights (θ_g):')
for term, w in zip(structure.terms, weights.theta_g):
    print(f'  {term.display_name:25s}  {w:+.4f}')

## Training metrics

In [ ]:
metrics = compute_metrics(
    features_list, chosen_indices,
    weights.get_weights, customer_ids, categories
)
print_metrics(metrics, label='Train')

## Validation metrics

Load validation choice events (from 02_dsl_features output).

In [ ]:
# Load val features (must rerun feature extraction on val set first)
# For now: run a quick check using the training data as proxy
val_metrics = compute_metrics(
    features_list[-500:], chosen_indices[-500:],
    weights.get_weights, customer_ids[-500:], categories[-500:]
)
print_metrics(val_metrics, label='Val (proxy)')

## Residual analysis

In [ ]:
# Build metadata list
with open(DATA_DIR / 'train_choices.pkl', 'rb') as f:
    train_choices = pickle.load(f)
metadata = [e['metadata'] for e in train_choices[:len(features_list)]]

residuals = compute_residuals(
    features_list, chosen_indices,
    weights.get_weights, customer_ids, categories, metadata
)
print('Residual breakdown:')
for k, v in residuals.items():
    print(f'  {k}: {v}')

## Weight visualization

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
colors = ['steelblue' if w >= 0 else 'tomato' for w in weights.theta_g]
ax.barh(structure.terms, weights.theta_g, color=colors)
ax.axvline(0, color='k', linewidth=0.8)
ax.set_xlabel('Weight (θ_g)')
ax.set_title(f'Global weights — {structure}')
plt.tight_layout()
plt.show()

## Posterior score formula

$$\text{Score}(S, \theta) = \log p(D|\theta,S) + \log p(\theta|S) + \log p(S)$$

The complexity prior penalizes longer structures: $\log p(S) = -L(S) \cdot \ln 2$

In [ ]:
print(f'L(S) = {structure.complexity()}  terms')
print(f'log p(S) = {structure.log_prior():.4f}')
print(f'Total posterior score = {score:.2f}')

# Save with checkpoint
ckpt.save('initial_fit', {
    'current_state.pkl': {
        'structure': structure.to_dict(),
        'score': score,
        'metrics': metrics,
        'residuals': residuals,
    }
}, ['current_state.pkl'])
print('\nSaved current_state.pkl for outer loop.')